In [ ]:
pip install rapidfuzz


In [ ]:
import pandas as pd
from io import BytesIO
from ipywidgets import FileUpload
# import sys
# import platform

def select_excel_file():
    uploader = FileUpload(accept='.xlsx,.xls,.xlsm', multiple=False)
    uploader
    return uploader

In [ ]:
excelfile_mata = select_excel_file()
excelfile_mata

In [ ]:
if not excelfile_mata.value:
    raise FileNotFoundError("No file uploaded")
val = excelfile_mata.value
if isinstance(val, dict):
    file_data = next(iter(val.values()))
elif isinstance(val, tuple):
    file_data = val[0]
else:
    raise TypeError("Unknown FileUpload.value format")
file_bytes = file_data["content"]

def read_excel_bytes(file_bytes):
    required_sheets = ["YgnTBPro","TPT Register"]

    with pd.ExcelFile(BytesIO(file_bytes), engine="openpyxl") as xf:
        try:
            dfs = pd.read_excel(
                        BytesIO(file_bytes),
                        sheet_name=required_sheets)
            for name in required_sheets:
               dfs[name]['SheetName'] = name
            return dfs
        except Exception:
            print("Available sheets:", xf.sheet_names)
            raise
df_mata = read_excel_bytes(file_bytes)


In [ ]:
dfmata = df_mata['YgnTBPro']
dfmata = dfmata[dfmata['VOL']==1]
dfmata_columns = ['PatientID','Name','Age','Sex','VOL','Referralfor','VolunteerName','Organization','TPTregno','RegnoofindexTBcase','TPTregistrationdate','TPTregimen','Date','TownshipTBRegNumber','TypeofTBTreatment','Team','Tsp','Approach','Visit_no','Sr_No']
dfmata = dfmata[dfmata_columns]
dfmata['Sex'] = dfmata['Sex'].replace({1:"M", 2:"F"})
dfmata['Referralfor'] = dfmata['Referralfor'].replace({1:"Presumptive", 2:"CI"})
dfmata['TypeofTBTreatment'] = dfmata['TypeofTBTreatment'].replace({1:"DS-TB", 2:"DR-TB",3:"TPT"})
# dfmata

In [ ]:
excelfilepgk = select_excel_file()
excelfilepgk

In [ ]:
if not excelfilepgk.value:
    raise FileNotFoundError("No file uploaded")
val = excelfilepgk.value
if isinstance(val, dict):
    file_data = next(iter(val.values()))
elif isinstance(val, tuple):
    file_data = val[0]
else:
    raise TypeError("Unknown FileUpload.value format")
file_bytes = file_data["content"]

def read_excel_bytes_pgk(file_bytes):
    required_sheets = ["Presumptive","CI"]

    with pd.ExcelFile(BytesIO(file_bytes), engine="openpyxl") as xf:
        try:
            dfs = pd.read_excel(
                        BytesIO(file_bytes),
                        sheet_name=required_sheets)
            for name in required_sheets:
               dfs[name]['SheetName'] = name
            return dfs
        except Exception:
            print("Available sheets:", xf.sheet_names)
            raise
df_pgk = read_excel_bytes_pgk(file_bytes)


In [ ]:
dfpgk_c = df_pgk['CI']
dfpgk_p = df_pgk['Presumptive']

In [ ]:
dfpgk_p.rename(columns={'Name of presumptive TB patient': 'Name','VL Name':'VolunteerName','TB Registeration No.':'TownshipTBRegNumber','Diagnosis Result ':'TypeofTBTreatment'}, inplace=True)       
dfpgk_c.rename(columns={'Sex':'PatientSex','Age':'PatientAge'},inplace=True)
dfpgk_c.rename(columns={'Name of Contact':'Name','Sex2':'Sex', 'Age3':'Age','Name':'VolunteerName','Result ':'TypeofTBTreatment'}, inplace=True)

In [ ]:
dfpgk_c['SheetName'] = 'CI'
dfpgk_p['SheetName'] = 'Presumptive'
dfpgk = pd.concat([dfpgk_c, dfpgk_p], ignore_index=True, sort=False)
dfpgk['TypeofTBTreatment'] = dfpgk['TypeofTBTreatment'].replace({'No active TB': pd.NA})


In [ ]:
dfpgk.rename(columns={'Patients ID':'PatientID'},inplace=True)

In [ ]:
dfpgk = dfpgk[dfpgk["PatientID"].astype(str).str.startswith("5")]

In [ ]:
import pandas as pd
import difflib
from rapidfuzz import fuzz

# Make copies
mata = dfmata.copy()
pgk = dfpgk.copy()

# Ensure PatientID is string
mata["PatientID"] = mata["PatientID"].astype(str).str.strip()
pgk["PatientID"] = pgk["PatientID"].astype(str).str.strip()

# Function to normalize names
def normalize_name(name):
    if pd.isna(name):
        return ""
    return "".join(str(name).split()).lower()

# Merge both dataframes
merged = mata.merge(
    pgk,
    on="PatientID",
    how="outer",
    suffixes=("_MATA", "_PGK"),
    indicator=True
)

results = []

for _, row in merged.iterrows():

    pid = row["PatientID"]

    # ID missing
    if row["_merge"] == "left_only":
        results.append({
            "PatientID": pid,
            "Difference": "PatientID missing in PGK",
            "PGK": "",
            "MATA": pid
        })
        continue

    if row["_merge"] == "right_only":
        results.append({
            "PatientID": pid,
            "Difference": "PatientID missing in MATA",
            "PGK": pid,
            "MATA": ""
        })
        continue

    # Compare Presumptive/CI
    referral_mata = row["Referralfor"]
    referral_pgk = row["SheetName"]
    if pd.isna(referral_mata) != pd.isna(referral_pgk) or (
        pd.notna(referral_mata) and pd.notna(referral_pgk) and str(referral_mata) != str(referral_pgk)):
        results.append({
            "PatientID": pid,
            "Difference": "PresumptiveCI",
            "PGK": referral_pgk,
            "MATA": referral_mata
        })

    # Compare Name
    name_mata = row["Name_MATA"]
    name_pgk = row["Name_PGK"]
    normalized_mata = normalize_name(name_mata)
    normalized_pgk = normalize_name(name_pgk)

    # score = difflib.SequenceMatcher(None, normalized_mata, normalized_pgk).ratio() * 100
    score = fuzz.token_sort_ratio(normalized_mata, normalized_pgk)

    if normalized_mata != normalized_pgk:
        results.append({
            "PatientID": pid,
            "Difference": "Name",
            "PGK": name_pgk,
            "MATA": name_mata,
            "Score": round(score, 1)
        })

    # Compare Sex
    sex_mata = row["Sex_MATA"]
    sex_pgk = row["Sex_PGK"]
    if pd.isna(sex_mata) != pd.isna(sex_pgk) or (
        pd.notna(sex_mata) and pd.notna(sex_pgk) and str(sex_mata) != str(sex_pgk)
    ):
        results.append({
            "PatientID": pid,
            "Difference": "Sex",
            "PGK": sex_pgk,
            "MATA": sex_mata
        })

    # Compare Age
    age_mata = row["Age_MATA"]
    age_pgk = row["Age_PGK"]
    if pd.isna(age_mata) != pd.isna(age_pgk) or (
        pd.notna(age_mata) and pd.notna(age_pgk) and age_mata != age_pgk
    ):
        results.append({
            "PatientID": pid,
            "Difference": "Age",
            "PGK": age_pgk,
            "MATA": age_mata
        })

    # Compare Volunteer Name
    volname_mata = row["VolunteerName_MATA"]
    volname_pgk = row["VolunteerName_PGK"]
    normalized_matavol = normalize_name(name_mata)
    normalized_pgkvol = normalize_name(name_pgk)

    # score = difflib.SequenceMatcher(None, normalized_matavol, normalized_pgkvol).ratio() * 100
    score = fuzz.token_sort_ratio(normalized_matavol, normalized_pgkvol)

    if normalized_matavol != normalized_pgkvol:
        results.append({
            "PatientID": pid,
            "Difference": "VolunteerName",
            "PGK": volname_pgk,
            "MATA": volname_mata,
            "Score": round(score, 1)
        })

    # Compare TypeofTBTreatment
    TypeofTBTreatment_mata = row["TypeofTBTreatment_MATA"]
    TypeofTBTreatment_pgk = row["TypeofTBTreatment_PGK"]
    if pd.isna(TypeofTBTreatment_mata) != pd.isna(TypeofTBTreatment_pgk) or (
        pd.notna(TypeofTBTreatment_mata) and pd.notna(TypeofTBTreatment_pgk) and str(TypeofTBTreatment_mata) != str(TypeofTBTreatment_pgk)
    ):
        results.append({
            "PatientID": pid,
            "Difference": "TypeofTBTreatment",
            "PGK": TypeofTBTreatment_pgk,
            "MATA": TypeofTBTreatment_mata
        })

    # Compare TownshipTBRegNumber
    TownshipTBRegNumber_mata = row["TownshipTBRegNumber_MATA"]
    TownshipTBRegNumber_pgk = row["TownshipTBRegNumber_PGK"]
    normalized_mata_TownshipTBRegNumber = normalize_name(TownshipTBRegNumber_mata)
    normalized_pgk_TownshipTBRegNumber = normalize_name(TownshipTBRegNumber_pgk)

    # score = difflib.SequenceMatcher(None, normalized_mata_TownshipTBRegNumber, normalized_pgk_TownshipTBRegNumber).ratio() * 100
    score = fuzz.token_sort_ratio(normalized_mata_TownshipTBRegNumber, normalized_pgk_TownshipTBRegNumber)

    if normalized_mata_TownshipTBRegNumber != normalized_pgk_TownshipTBRegNumber:
        results.append({
            "PatientID": pid,
            "Difference": "TownshipTBRegNumber",
            "PGK": TownshipTBRegNumber_pgk,
            "MATA": TownshipTBRegNumber_mata,
            "Score": round(score, 1)
        })


# Final result dataframe
df_difference = pd.DataFrame(results)

# Display
# print(df_difference)

# Optional: save to Excel
# df_difference.to_excel("Data_Differences.xlsx", index=False)

In [ ]:
from datetime import datetime
dataframes = {'CheckData': df_difference}
timestamp_create = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
output_file_name = fr"PythonDataVerification_{timestamp_create}"
file_path = fr"C:\Users\user\Downloads\vsoutput_{output_file_name}.xlsx"  ### WINDOW
# file_path = fr"/Users/toepyaesone/Downloads/vsoutput_{output_file_name}.xlsx"  ### MAC
#file_path = fr"vsoutput_{output_file_name}.xlsx"    ### IOS
with pd.ExcelWriter(file_path) as writer:
    for sheet_name, df in dataframes.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sheet_name, index=False)
        else:
            print(f"Skipping empty or None DataFrame for sheet '{sheet_name}'.")
print(file_path)